# 04 — Saliency Maps (Captum)
Compute token-level attribution using LayerIntegratedGradients.
Includes side-by-side comparison of baseline vs LoRA attention patterns.

In [ ]:
import os
import sys
import pathlib

IN_COLAB = 'google.colab' in str(get_ipython())

PACE_PROJECT_ROOT = '/home/hice1/arios35/scratch/slm_logic_hardening'

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)
    OUTPUT_BASE = '/content/drive/MyDrive/slm_logic_hardening/outputs'
    os.chdir('/content/drive/MyDrive/slm_logic_hardening')
    sys.path.insert(0, '/content/drive/MyDrive/slm_logic_hardening')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'peft', 'datasets', 'captum', 'sentencepiece', 'pyyaml'])
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
else:
    # PACE-ICE or local
    if os.path.exists(PACE_PROJECT_ROOT):
        project_root = pathlib.Path(PACE_PROJECT_ROOT)
    else:
        project_root = pathlib.Path().resolve()
        if not (project_root / 'scripts').exists():
            project_root = project_root.parent
    os.chdir(str(project_root))
    sys.path.insert(0, str(project_root))
    OUTPUT_BASE = str(project_root / 'outputs')

    # HF token — from env var or prompt
    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        import getpass
        hf_token = getpass.getpass('Enter HF token: ')
    from huggingface_hub import login
    login(token=hf_token)

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
print(f'Working dir: {os.getcwd()}')
print(f'Output base: {OUTPUT_BASE}')

In [ ]:
import torch
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## Configuration

In [ ]:
import yaml
from pathlib import Path

MODEL_KEY = 'phi35'

# Load indices and n_steps from config — single source of truth
with open('configs/eval/saliency.yml') as f:
    saliency_cfg = yaml.safe_load(f)

SAMPLE_INDICES = saliency_cfg['sample_indices']
N_STEPS = saliency_cfg['n_steps']

print(f'Sample indices ({len(SAMPLE_INDICES)}): {SAMPLE_INDICES}')
print(f'n_steps: {N_STEPS}')

# Experiment directories
BASELINE_DIR = Path(OUTPUT_BASE) / 'phi35' / 'phi35_baseline'
LORA_DIR     = Path(OUTPUT_BASE) / 'phi35' / 'phi35_lora_folio'

LORA_ADAPTER = str(LORA_DIR / 'final_adapter')

print(f'\nBaseline dir: {BASELINE_DIR}')
print(f'LoRA dir:     {LORA_DIR}')
print(f'LoRA adapter: {LORA_ADAPTER}')

Install Captum

In [ ]:
!pip install -q transformers peft datasets captum sentencepiece pyyaml

## 1. Compute baseline saliency

In [ ]:
from scripts.saliency import run_saliency

run_saliency(
    model_key=MODEL_KEY,
    mode='baseline',
    adapter_path=None,
    sample_indices=SAMPLE_INDICES,
    n_steps=N_STEPS,
    output_dir=BASELINE_DIR,
    experiment_id='phi35_baseline',
)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Map: 100%|##########| 1001/1001 [00:00<?, ? examples/s]

Map: 100%|##########| 203/203 [00:00<?, ? examples/s]


Example 0: ground_truth = Uncertain
Computing attributions...
Top 10 tokens:
  ▁Log                  132.114822
  ▁Re                   100.435028
  :                     68.404289
  ical                  64.404991
  ▁are                  52.083549
  oth                   51.973244
  one                   51.248520
  one                   40.722733
  :                     40.299137
  yp                    37.591328

Example 2: ground_truth = False
Computing attributions...
Top 10 tokens:
  ▁academic             191.201752
  :                     102.498161
  ▁Re                   73.814865
  ?                     71.196159
  ▁the                  68.489067
  :                     67.018921
  <0x0A>                56.162571
  ical                  54.263298
  ▁a                    41.021954
  ▁Log                  39.747364

Example 4: ground_truth = Uncertain
Computing attributions...
Top 10 tokens:
  ▁Re                   161.810730
  ical                  137.988556
  ▁Log          

## 2. Compute LoRA saliency

In [ ]:
from scripts.saliency import run_saliency
run_saliency(
    model_key=MODEL_KEY,
    mode='lora',
    adapter_path=LORA_ADAPTER,
    sample_indices=SAMPLE_INDICES,
    n_steps=N_STEPS,
    output_dir=LORA_DIR,
    experiment_id='phi35_lora_folio',
    #experiment_id='phi35_lora_folio_pw',
    #experiment_id='phi35_lora_folio_pw_rt',
)

## 3. Side-by-side comparison (before vs after fine-tuning)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def load_saliency_dir(saliency_dir: Path, indices: list) -> dict:
    results = {}
    for idx in indices:
        f = saliency_dir / 'saliency' / f'{idx}.json'
        if f.exists():
            with open(f) as fp:
                results[idx] = json.load(fp)
    return results

def top_n(attributions, n=10):
    return sorted(attributions, key=lambda x: x['score'], reverse=True)[:n]

baseline_sal = load_saliency_dir(BASELINE_DIR, SAMPLE_INDICES)
lora_sal     = load_saliency_dir(LORA_DIR, SAMPLE_INDICES)

print(f'Baseline loaded: {len(baseline_sal)} | LoRA loaded: {len(lora_sal)}')

# LORA_LABEL controls the chart title — update when switching models
LORA_LABEL = 'Phi-3.5 + LoRA (FOLIO)'
#LORA_LABEL = 'Phi-3.5 + LoRA (FOLIO+PW)'
#LORA_LABEL = 'Phi-3.5 + LoRA (FOLIO+PW+RT)'

for idx in SAMPLE_INDICES:
    if idx not in baseline_sal or idx not in lora_sal:
        print(f'Skipping {idx} — not computed yet')
        continue

    b_res = baseline_sal[idx]
    l_res = lora_sal[idx]
    label = b_res['ground_truth']
    premises = b_res.get('no_of_premises', '?')

    b_top = top_n(b_res['attributions'])
    l_top = top_n(l_res['attributions'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'Example {idx} — GT: {label}  |  Premises: {premises}', fontsize=13)

    for ax, top, title in [
        (axes[0], b_top, 'Phi-3.5 Baseline'),
        (axes[1], l_top, LORA_LABEL),
    ]:
        tokens = [t['token'] for t in top]
        scores = [t['score'] for t in top]
        y_pos = np.arange(len(tokens))
        ax.barh(y_pos, scores, color='#4C72B0')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(tokens)
        ax.invert_yaxis()
        ax.set_title(title)
        ax.set_xlabel('Attribution Score')

    plt.tight_layout()
    out = BASELINE_DIR.parent / f'saliency_comparison_{idx}.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'Saved: {out}')